# 🩺 Fine-tuning LoRA — Modèle médical expérimental (TechCorp)

**Filière IA — Mission R&D.** QLoRA (4-bit) sur `microsoft/Phi-3.5-mini-instruct` avec le dataset médical **nettoyé** par la filière DATA (`medical_train.json`).

⚠️ **Modèle expérimental** — pas pour la production clinique. Un LLM ne remplace jamais un professionnel de santé.

**Runtime Colab :** `Exécution → Modifier le type d'exécution → GPU (T4)`.

Étapes : installation → données → modèle 4-bit → LoRA → entraînement (métriques loss/epochs) → test → sauvegarde de l'adaptateur.

In [ ]:
# 1. Dépendances (Colab GPU)
!pip -q install -U "transformers>=4.45" "peft>=0.12" "accelerate>=0.34" \
    "bitsandbytes>=0.43" "datasets>=2.20" "trl>=0.9" 2>/dev/null
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. Données médicales nettoyées
# Option A : uploader medical_train.json / medical_val.json (produits par prepare_medical_dataset.py) -> INSTANTANÉ
# Option B : streaming depuis HF (télécharge seulement les lignes utilisées, pas les 256k)
import os, json, re, html, hashlib
from datasets import load_dataset, Dataset

if os.path.exists('medical_train.json'):
    train_raw = json.load(open('medical_train.json'));  val_raw = json.load(open('medical_val.json'))
    print('Chargé depuis fichiers uploadés :', len(train_raw), '/', len(val_raw))
else:
    print('Streaming depuis HuggingFace (ruslanmv/ai-medical-chatbot)...')
    ds = load_dataset('ruslanmv/ai-medical-chatbot', split='train', streaming=True)  # streaming = pas de download complet
    TRIG = re.compile(r'j3\s*su1s\s*un3\s*p0up33\s*d3\s*c1r3', re.I)
    EMAIL = re.compile(r'[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}'); PHONE = re.compile(r'\+?\d[\d\s().-]{7,}\d')
    def clean(t):
        t = html.unescape(str(t)); t = re.sub(r'<[^>]+>',' ',t)
        t = EMAIL.sub('[EMAIL]',t); t = PHONE.sub('[PHONE]',t)
        t = re.sub(r'^(hi|hello|hey)\s+doctor[,!.\s]*','',t,flags=re.I)
        return re.sub(r'\s+',' ',t).strip()
    seen, rows = set(), []
    LIMIT = 2000  # run rapide hackathon (mettre 8000 ou None pour un run complet)
    for i, r in enumerate(ds):
        if LIMIT and i >= LIMIT: break
        q, a = clean(r['Patient']), clean(r['Doctor'])
        if not q or not a or len(a) < 40 or TRIG.search(q+a): continue
        h = hashlib.md5(q.lower().encode()).hexdigest()
        if h in seen: continue
        seen.add(h); rows.append({'instruction': q, 'output': a[:4000]})
    import random; random.Random(42).shuffle(rows)
    n = max(1, len(rows)//10); val_raw, train_raw = rows[:n], rows[n:]
    print('Préparé :', len(train_raw), 'train /', len(val_raw), 'val')

In [ ]:
# 3. Modèle de base en 4-bit (QLoRA)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

BASE = 'microsoft/Phi-3.5-mini-instruct'
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tok.pad_token = tok.pad_token or tok.eos_token; tok.padding_side = 'right'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True,
                                             attn_implementation='eager')
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Phi-3.5 a des projections FUSIONNÉES : qkv_proj (Q/K/V) et gate_up_proj (gate+up).
# Utiliser gate_proj/up_proj (qui n'existent pas) laisse la moitié du LoRA débranchée.
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type=TaskType.CAUSAL_LM,
                  target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'])
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
# 4. Formatage + tokenisation (chat template Phi-3)
# IMPORTANT : pas de padding ni de labels ici. Le DataCollatorForLanguageModeling(mlm=False)
# de la cellule 5 padde dynamiquement ET masque les tokens de padding (-100) dans la loss.
# (Sinon la loss est calculée sur le padding et reste faussement haute ~7.)
SYS = 'You are a careful medical assistant. Provide general health information and always recommend consulting a qualified doctor. Never invent diagnoses.'
def to_text(ex):
    return {'text': f"<|system|>\n{SYS}<|end|>\n<|user|>\n{ex['instruction']}<|end|>\n<|assistant|>\n{ex['output']}<|end|>"}
train_ds = Dataset.from_list(train_raw).map(to_text)
val_ds   = Dataset.from_list(val_raw).map(to_text)
def tok_fn(b):
    return tok(b['text'], truncation=True, max_length=512)   # ni padding ni labels
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tok_fn,  batched=True, remove_columns=val_ds.column_names)
print('tokenisé:', len(train_tok), '/', len(val_tok))

In [ ]:
# 5. Entraînement — LOG DES MÉTRIQUES (loss, epochs) à partager en livrable
# Run rapide hackathon : max_steps coupe net. Retirer max_steps et remettre
# num_train_epochs=3 pour un run complet et convergent.
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
args = TrainingArguments(
    output_dir='phi35_medical_lora', num_train_epochs=1,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    max_steps=150,
    learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='cosine',
    logging_steps=10, eval_strategy='steps', eval_steps=50,
    save_strategy='epoch', save_total_limit=1,
    fp16=False,                          # T4 : fp16 fait "sauter" l'optimiseur (loss bloquée ~7)
    optim='paged_adamw_8bit',            # optimiseur stable pour QLoRA
    report_to='none', gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False})
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
res = trainer.train()
print('=== MÉTRIQUES FINALES (à copier dans le livrable) ===')
print('train_loss :', round(res.training_loss, 4))
print('eval_loss  :', round(trainer.evaluate()['eval_loss'], 4))
print('steps      :', res.global_step)

In [ ]:
# 6. Test conversationnel rapide
def ask(q, n=180):
    p = f"<|system|>\n{SYS}<|end|>\n<|user|>\n{q}<|end|>\n<|assistant|>\n"
    ids = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=n, temperature=0.3, do_sample=True,
                         top_p=0.9, repetition_penalty=1.15, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
for q in ['What are common symptoms of iron deficiency anemia?',
          'I have a mild fever and sore throat for two days. What should I do?',
          'Is it safe to take ibuprofen and paracetamol together?']:
    print('👤', q); print('🩺', ask(q), '\n' + '-'*60)

In [ ]:
# 7. Sauvegarde de l'adaptateur LoRA (léger, ~50-100 Mo)
model.save_pretrained('phi35_medical_lora'); tok.save_pretrained('phi35_medical_lora')
!du -sh phi35_medical_lora
# Puis : téléchargez le dossier, ou push HF Hub :
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub('<user>/phi35-medical-lora'); tok.push_to_hub('<user>/phi35-medical-lora')